# Chapter 28 — MCMC: Markov Chain Monte Carlo
*Tier 3: Data Scientists & Students*

## 🎯 Learning Objectives
- Understand why direct sampling from complex posteriors is hard
- Implement Metropolis-Hastings and Gibbs sampling
- Apply MCMC to Bayesian parameter estimation

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import scipy.stats as stats

rng = np.random.default_rng(42)
plt.style.use("seaborn-v0_8-whitegrid")
print("Libraries loaded ✅")

## 1. Concept Review — Why MCMC?

For Bayesian inference, we want samples from:
$$p(\theta | \mathcal D) \propto p(\mathcal D | \theta) \cdot p(\theta)$$

When this has no closed form (most real problems), we use **MCMC**:
construct a Markov chain whose **stationary distribution = target posterior**.

Key algorithms:
- **Metropolis-Hastings** (MH): general purpose, requires proposal distribution
- **Gibbs sampling**: samples each parameter conditional on others
- **HMC / NUTS**: gradient-based (used by Stan, PyMC)

In [ ]:
# Metropolis-Hastings for a bimodal distribution
def log_target(x):
    # Log density of bimodal: 0.3*N(-3,1) + 0.7*N(2,0.8)
    return np.log(0.3*stats.norm.pdf(x, -3, 1) + 0.7*stats.norm.pdf(x, 2, 0.8) + 1e-300)

def metropolis_hastings(n_samples, proposal_std=1.0, x0=0.0):
    samples = np.zeros(n_samples)
    x = x0
    n_accept = 0
    for i in range(n_samples):
        x_prop = x + rng.normal(0, proposal_std)
        log_ratio = log_target(x_prop) - log_target(x)
        if np.log(rng.random()) < log_ratio:
            x = x_prop
            n_accept += 1
        samples[i] = x
    return samples, n_accept/n_samples

samples, accept_rate = metropolis_hastings(50_000, proposal_std=1.5)
burn_in = 1000
samples_post = samples[burn_in:]
print(f"Acceptance rate: {accept_rate:.3f}")

x_range = np.linspace(-7, 6, 400)
target_pdf = 0.3*stats.norm.pdf(x_range,-3,1) + 0.7*stats.norm.pdf(x_range,2,0.8)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(samples[:500], alpha=0.7, lw=0.8)
axes[0].axvline(burn_in, color="red", lw=1, label="Burn-in end")
axes[0].set_title("MH Trace (first 500 steps)"); axes[0].legend()

axes[1].hist(samples_post, bins=100, density=True, alpha=0.6, label="MH samples")
axes[1].plot(x_range, target_pdf, "r-", lw=2, label="True target")
axes[1].set_title("MH Posterior vs True Target"); axes[1].legend()
plt.tight_layout(); plt.show()

## 2. Math Walkthrough — Detailed Balance

MH satisfies **detailed balance** (→ stationary distribution = target):
$$\pi(x)q(x'|x)\alpha(x,x') = \pi(x')q(x|x')\alpha(x',x)$$

where the acceptance ratio:
$$\alpha(x,x') = \min\left(1,\; \frac{\pi(x')q(x|x')}{\pi(x)q(x'|x)}\right)$$

In [ ]:
# Gibbs sampling for Bayesian linear regression
# y = beta0 + beta1*x + eps, eps~N(0, sigma^2)
# Priors: beta~N(0,100), sigma^2~InvGamma(1,1)

n_data = 50
x_data = rng.uniform(-3, 3, n_data)
y_data = 2 + 1.5*x_data + rng.normal(0, 1, n_data)

X_mat = np.column_stack([np.ones(n_data), x_data])

# Gibbs sampler (conjugate Normal-InvGamma)
n_mcmc = 10_000; burnin = 2000
beta_samples = np.zeros((n_mcmc, 2))
sig2_samples = np.zeros(n_mcmc)

beta = np.array([0.0, 0.0]); sig2 = 1.0
prior_var = 100.0

for i in range(n_mcmc):
    # Sample beta | sigma^2, data
    V_post = np.linalg.inv(X_mat.T @ X_mat / sig2 + np.eye(2) / prior_var)
    m_post = V_post @ (X_mat.T @ y_data / sig2)
    beta = rng.multivariate_normal(m_post, V_post)
    # Sample sigma^2 | beta, data
    resid = y_data - X_mat @ beta
    a_post = 1 + n_data/2
    b_post = 1 + resid @ resid / 2
    sig2 = 1/rng.gamma(a_post, 1/b_post)
    beta_samples[i] = beta
    sig2_samples[i] = sig2

b0_post = beta_samples[burnin:, 0]
b1_post = beta_samples[burnin:, 1]
s2_post = sig2_samples[burnin:]
print(f"β₀: mean={b0_post.mean():.3f}, 95% CI=[{np.percentile(b0_post,2.5):.3f}, {np.percentile(b0_post,97.5):.3f}] (true=2)")
print(f"β₁: mean={b1_post.mean():.3f}, 95% CI=[{np.percentile(b1_post,2.5):.3f}, {np.percentile(b1_post,97.5):.3f}] (true=1.5)")
print(f"σ²: mean={s2_post.mean():.3f}, 95% CI=[{np.percentile(s2_post,2.5):.3f}, {np.percentile(s2_post,97.5):.3f}] (true=1)")

In [ ]:
# Convergence diagnostics
fig, axes = plt.subplots(2, 2, figsize=(13, 7))
# Trace plots
axes[0,0].plot(b0_post[:2000], lw=0.5); axes[0,0].set_title("β₀ Trace")
axes[0,1].plot(b1_post[:2000], lw=0.5); axes[0,1].set_title("β₁ Trace")
# Posteriors
axes[1,0].hist(b0_post, bins=60, density=True, alpha=0.7)
axes[1,0].axvline(2, color="red", lw=2, label="True β₀=2"); axes[1,0].legend()
axes[1,0].set_title("β₀ Posterior")
axes[1,1].hist(b1_post, bins=60, density=True, alpha=0.7)
axes[1,1].axvline(1.5, color="red", lw=2, label="True β₁=1.5"); axes[1,1].legend()
axes[1,1].set_title("β₁ Posterior")
plt.tight_layout(); plt.show()